In [0]:
-- Business Question 3: Zone-Level Mobility Patterns & Opportunities
-- Grain: Day of week + hour + taxi zone + weather condition
-- Metrics: Pickups, dropoffs, net flow, fare, distance, duration
--
-- Pickup activity uses the actual pickup date and hour.
-- Dropoff activity uses the actual dropoff date and hour.
--
-- IMPORTANT:
-- The fact table stores weather associated with pickup time.
-- Therefore, weather fields attached to dropoff activity represent
-- the pickup weather of those trips, not weather at dropoff time.
--
-- Fare, distance, and duration metrics are attributed to both pickup
-- and dropoff zone activity. Do not sum these metrics across all zones
-- to obtain systemwide totals.

WITH pickup_activity AS (
    SELECT
        d.day_of_week,
        d.day_name,
        h.hour_of_day AS hour,
        h.day_period AS time_of_day,

        w.temperature_band,
        w.weather_condition,
        w.is_raining AS precipitation_flag,

        t.pickup_zone_key AS zone_key,

        SUM(t.trip_count) AS pickups,
        0 AS dropoffs,

        SUM(t.fare_amount) AS total_fare,
        COUNT(t.fare_amount) AS fare_count,

        SUM(t.trip_distance) AS total_distance,
        COUNT(t.trip_distance) AS distance_count,

        SUM(t.trip_duration_minutes) AS total_duration_minutes,
        COUNT(t.trip_duration_minutes) AS duration_count

    FROM nyc_mobility.gold.fact_trip AS t

    JOIN nyc_mobility.gold.dim_date AS d
        ON t.pickup_date_key = d.date_key

    JOIN nyc_mobility.gold.dim_hour AS h
        ON t.pickup_hour_key = h.hour_key

    JOIN nyc_mobility.gold.dim_weather AS w
        ON t.weather_key = w.weather_key

    GROUP BY
        d.day_of_week,
        d.day_name,
        h.hour_of_day,
        h.day_period,
        w.temperature_band,
        w.weather_condition,
        w.is_raining,
        t.pickup_zone_key
),

dropoff_activity AS (
    SELECT
        d.day_of_week,
        d.day_name,
        h.hour_of_day AS hour,
        h.day_period AS time_of_day,

        -- Weather represents pickup weather because fact_trip
        -- does not store a separate dropoff weather key.
        w.temperature_band,
        w.weather_condition,
        w.is_raining AS precipitation_flag,

        t.dropoff_zone_key AS zone_key,

        0 AS pickups,
        SUM(t.trip_count) AS dropoffs,

        SUM(t.fare_amount) AS total_fare,
        COUNT(t.fare_amount) AS fare_count,

        SUM(t.trip_distance) AS total_distance,
        COUNT(t.trip_distance) AS distance_count,

        SUM(t.trip_duration_minutes) AS total_duration_minutes,
        COUNT(t.trip_duration_minutes) AS duration_count

    FROM nyc_mobility.gold.fact_trip AS t

    JOIN nyc_mobility.gold.dim_date AS d
        ON t.dropoff_date_key = d.date_key

    JOIN nyc_mobility.gold.dim_hour AS h
        ON t.dropoff_hour_key = h.hour_key

    JOIN nyc_mobility.gold.dim_weather AS w
        ON t.weather_key = w.weather_key

    GROUP BY
        d.day_of_week,
        d.day_name,
        h.hour_of_day,
        h.day_period,
        w.temperature_band,
        w.weather_condition,
        w.is_raining,
        t.dropoff_zone_key
),

combined_activity AS (
    SELECT * FROM pickup_activity

    UNION ALL

    SELECT * FROM dropoff_activity
),

aggregated AS (
    SELECT
        a.day_of_week,
        a.day_name,
        a.hour,
        a.time_of_day,
        a.temperature_band,
        a.weather_condition,
        a.precipitation_flag,
        a.zone_key,

        z.borough,
        z.zone,

        SUM(a.pickups) AS pickups,
        SUM(a.dropoffs) AS dropoffs,

        SUM(a.dropoffs) - SUM(a.pickups) AS net_flow,

        ABS(
            SUM(a.dropoffs) - SUM(a.pickups)
        ) AS absolute_flow_imbalance,

        SUM(a.total_fare) AS total_fare,
        SUM(a.fare_count) AS fare_count,

        SUM(a.total_distance) AS total_distance,
        SUM(a.distance_count) AS distance_count,

        SUM(a.total_duration_minutes) AS total_duration_minutes,
        SUM(a.duration_count) AS duration_count

    FROM combined_activity AS a

    JOIN nyc_mobility.gold.dim_zone AS z
        ON a.zone_key = z.zone_key

    GROUP BY
        a.day_of_week,
        a.day_name,
        a.hour,
        a.time_of_day,
        a.temperature_band,
        a.weather_condition,
        a.precipitation_flag,
        a.zone_key,
        z.borough,
        z.zone
)

SELECT
    day_of_week,
    day_name,
    hour,
    time_of_day,

    temperature_band,
    weather_condition,
    precipitation_flag,

    zone_key,
    borough,
    zone,

    pickups,
    dropoffs,

    net_flow,
    absolute_flow_imbalance,

    pickups + dropoffs AS total_zone_activity,

    total_fare,
    total_distance,
    total_duration_minutes,

    ROUND(
        total_distance / NULLIF(distance_count, 0),
        2
    ) AS avg_trip_distance,

    ROUND(
        total_fare / NULLIF(fare_count, 0),
        2
    ) AS avg_fare,

    ROUND(
        total_duration_minutes / NULLIF(duration_count, 0),
        2
    ) AS avg_trip_duration_minutes

FROM aggregated

ORDER BY
    absolute_flow_imbalance DESC;